In [1]:
%pip install mediapipe opencv-python pandas numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
print(sys.executable)

d:\anaconda3\python.exe


In [ ]:
!pip list

In [3]:
!pip show protobuf

Name: protobuf
Version: 4.25.3
Summary: 
Home-page: https://developers.google.com/protocol-buffers/
Author: protobuf@googlegroups.com
Author-email: protobuf@googlegroups.com
License: 3-Clause BSD License
Location: C:\Users\Manav Pathak\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: 
Required-by: kaggle, mediapipe, streamlit, tensorboard, tensorflow


In [5]:
import sys

print(f"Python version: {sys.version}")
print(f"Version info: {sys.version_info}")
print(f"Major version: {sys.version_info.major}")
print(f"Minor version: {sys.version_info.minor}")
print(f"Micro version: {sys.version_info.micro}")

# Check for minimum version requirement
if sys.version_info >= (3, 8):
    print("Python 3.8 or higher detected")
else:
    print("Python version is too old")

Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Version info: sys.version_info(major=3, minor=12, micro=7, releaselevel='final', serial=0)
Major version: 3
Minor version: 12
Micro version: 7
Python 3.8 or higher detected


In [2]:
import mediapipe as mp
import cv2
import csv
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

In [3]:
cap = cv2.VideoCapture(0)   ## 3) landmark detection
# Initiate holistic model

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False 
        
        # Make Detections  *********
        results = holistic.process(image)
        # print(results.face_landmarks)
        
        # face_landmarks, pose_landmarks, left_hand_landmarks, right_hand_landmarks
        
        # Recolor image back to BGR for rendering
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        image.flags.writeable = True
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS, 
                                 mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                                 mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                                 )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                                 )

        # 3. Left Hand
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                                 )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                 )
        

        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# csv make`

In [ ]:
# (Optionally, use default values if landmarks are missing)
pose_count = len(results.pose_landmarks.landmark) if results.pose_landmarks else 33
face_count = len(results.face_landmarks.landmark) if results.face_landmarks else 468
right_hand_count = len(results.right_hand_landmarks.landmark) if results.right_hand_landmarks else 21
left_hand_count = len(results.left_hand_landmarks.landmark) if results.left_hand_landmarks else 21

# Start with a 'class' label (if you're using it for classification)
landmarks = ['class']

# Pose landmarks (x, y, z, v)
for i in range(1, pose_count + 1):
    landmarks += [f'pose_x{i}', f'pose_y{i}', f'pose_z{i}', f'pose_v{i}']

# Face landmarks (x, y, z, v)
for i in range(1, face_count + 1):
    landmarks += [f'face_x{i}', f'face_y{i}', f'face_z{i}', f'face_v{i}']

# Right hand landmarks (x, y, z)
for i in range(1, right_hand_count + 1):
    landmarks += [f'right_x{i}', f'right_y{i}', f'right_z{i}']

# Left hand landmarks (x, y, z)
for i in range(1, left_hand_count + 1):
    landmarks += [f'left_x{i}', f'left_y{i}', f'left_z{i}']

# Write header to CSV file
with open('coords_hands.csv', mode='w', newline='') as f:
    csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    csv_writer.writerow(landmarks)

In [ ]:
class_name = "happy"

In [ ]:

cap = cv2.VideoCapture(0)

# Initialize holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False        
        
        # Make Detections
        results = holistic.process(image)
        
        # Recolor image back to BGR for rendering
        image.flags.writeable = True   
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS, 
                                 mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                                 mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
        
        # 2. Right hand
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))

        # 3. Left Hand
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))

        # 4. Pose Detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

        # Export coordinates
        try:
            # Extract Pose landmarks
            pose = results.pose_landmarks.landmark if results.pose_landmarks else []
            pose_row = list(np.array([[landmark.x, landmark.y, landmark.z, landmark.visibility] for landmark in pose]).flatten())

            # Extract Face landmarks
            face = results.face_landmarks.landmark if results.face_landmarks else []
            face_row = list(np.array([[landmark.x, landmark.y, landmark.z, landmark.visibility] for landmark in face]).flatten())

            # Extract Right Hand landmarks
            right_hand = results.right_hand_landmarks.landmark if results.right_hand_landmarks else []
            right_hand_row = list(np.array([[landmark.x, landmark.y, landmark.z] for landmark in right_hand]).flatten())

            # Extract Left Hand landmarks
            left_hand = results.left_hand_landmarks.landmark if results.left_hand_landmarks else []
            left_hand_row = list(np.array([[landmark.x, landmark.y, landmark.z] for landmark in left_hand]).flatten())

            # Concatenate all rows
            row = pose_row + face_row + right_hand_row + left_hand_row

            # Append class name 
            row.insert(0, class_name)

            # Export to CSV
            with open('coords_hands.csv', mode='a', newline='') as f:
                csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                csv_writer.writerow(row) 

        except Exception as e:
            print("Error:", e)  # Print any errors
                        
        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


# 3)training

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the CSV file with hand landmarks
df = pd.read_csv('coords_hands.csv')
#df.head()  # Display the first few rows of the dataframe


In [ ]:
# Separate features and target
X = df.drop('class', axis=1)  # features
y = df['class']               # target value

# Handle missing values (fill missing values with 0)
X = X.fillna(0)

#X.head()  # Verify that there are no missing values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)


In [ ]:
y_test

## training

In [ ]:
from sklearn.pipeline import make_pipeline 
from sklearn.preprocessing import StandardScaler 

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

pipelines = {
    'lr': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    'rc': make_pipeline(StandardScaler(), RidgeClassifier()),
    'rf': make_pipeline(StandardScaler(), RandomForestClassifier()),
    'gb': make_pipeline(StandardScaler(), GradientBoostingClassifier()),
}

In [ ]:
fit_models = {}  # Dictionary to store fitted models

for algo, pipeline in pipelines.items():
    model = pipeline.fit(X_train, y_train)
    fit_models[algo] = model

In [ ]:
#predictions = fit_models['rc'].predict(X_test)
#predictions  # Display the predictions

In [ ]:
from sklearn.metrics import accuracy_score # Accuracy metrics 
import pickle 

In [ ]:
for algo, model in fit_models.items():
    yhat = model.predict(X_test)
    print(algo, accuracy_score(y_test, yhat))

In [ ]:
fit_models['rf'].predict(X_test)

In [ ]:
y_test

In [ ]:
with open('body_language_hands.pkl', 'wb') as f:
    pickle.dump(fit_models['rf'], f)

# 4) Final detection

In [4]:
from sklearn.metrics import accuracy_score # Accuracy metrics 
import pickle 

In [6]:
with open('body_language_hands.pkl', 'rb') as f:
    model = pickle.load(f)

In [7]:
model

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('randomforestclassifier', RandomForestClassifier())])

In [8]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, 
                          min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False        
        
        # Make Detections
        results = holistic.process(image)
        
        # Recolor image back to BGR for rendering
        image.flags.writeable = True   
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS, 
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )

        # 3. Left Hand
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )
        
        # Export coordinates and make predictions
        try:
            # --- Extract Pose Landmarks (x, y, z, visibility) ---
            if results.pose_landmarks is not None:
                
                pose_row = list(np.array([
                    [landmark.x, landmark.y, landmark.z, landmark.visibility]
                    for landmark in results.pose_landmarks.landmark
                ]).flatten())
            else:
                pose_row = [0] * (33 * 4)  # 33 landmarks × 4 values

            # --- Extract Face Landmarks (x, y, z, visibility) ---
            if results.face_landmarks is not None:
                face_row = list(np.array([
                    [landmark.x, landmark.y, landmark.z, landmark.visibility]
                    for landmark in results.face_landmarks.landmark
                ]).flatten())
            else:
                face_row = [0] * (468 * 4)  # 468 landmarks × 4 values

            # --- Extract Right Hand Landmarks (x, y, z) ---
            if results.right_hand_landmarks is not None:
                right_hand_row = list(np.array([
                    [landmark.x, landmark.y, landmark.z]
                    for landmark in results.right_hand_landmarks.landmark
                ]).flatten())
            else:
                right_hand_row = [0] * (21 * 3)  # 21 landmarks × 3 values

            # --- Extract Left Hand Landmarks (x, y, z) ---
            if results.left_hand_landmarks is not None:
                left_hand_row = list(np.array([
                    [landmark.x, landmark.y, landmark.z]
                    for landmark in results.left_hand_landmarks.landmark
                ]).flatten())
            else:
                left_hand_row = [0] * (21 * 3)  # 21 landmarks × 3 values

            # Concatenate rows in the order: pose, face, right hand, left hand
            row = pose_row + face_row + right_hand_row + left_hand_row

            # Prepare data for prediction (model.feature_names_in_ should match the CSV columns)
            X = pd.DataFrame([row], columns=model.feature_names_in_)
            body_language_class = model.predict(X)[0]
            body_language_prob = model.predict_proba(X)[0]

            # --- Grab ear coordinates for overlay (if pose is detected) ---
            if results.pose_landmarks is not None:
                left_ear = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_EAR]
                coords = tuple(np.multiply(np.array([left_ear.x, left_ear.y]), [640, 480]).astype(int))
            else:
                coords = (50, 50)  # Default coordinates if no pose is detected
            
            # Draw prediction rectangle and texts (unchanged styling)
            cv2.rectangle(
                image, 
                (coords[0], coords[1]+5), 
                (coords[0] + len(body_language_class)*20, coords[1]-30), 
                (245,117,16), -1
            )
            cv2.putText(
                image, body_language_class, coords, 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA
            )
            
            # Get status box
            cv2.rectangle(image, (0,0), (250,60), (245,117,16), -1)
            
            # Display Class
            cv2.putText(
                image, 'CLASS', (95,12), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA
            )
            cv2.putText(
                image, body_language_class.split(' ')[0], (90,40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA
            )
            
            # Display Probability
            cv2.putText(
                image, 'PROB', (15,12), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA
            )
            cv2.putText(
                image, str(round(body_language_prob[np.argmax(body_language_prob)],2)), (10,40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA
            )
            
        except Exception as e:
            print("Error:", e)  # Print any errors
                        
        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# Check if the camera opened successfully
if not cap.isOpened():
    print("Error: Could not open camera")
else:
    print("Camera opened successfully")

# Release the camera
cap.release()
cv2.destroyAllWindows()

print("Camera released successfully")
